In [1]:
import os
import pandas as pd
import numpy as np
import krippendorff
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters

In [ ]:
# 1. Define file paths and categories
file_paths = [
    "../0_data/annotator1.xlsx", 
    "annotator2.xlsx", 
    "annotator3.xlsx"
]

categories = [
    "Economic",
    "Capacity and resources",
    "Morality",
    "Fairness and equality",
    "Legality, constitutionality and jurisprudence",
    "Policy prescription and evaluation",
    "Crime and punishment",
    "Security and defense",
    "Health and safety",
    "Quality of life",
    "Cultural identity",
    "Public opinion",
    "Political",
    "External regulation and reputation"
]

# 2. Load Excel files (assumes rows are aligned across all files)
dfs = [pd.read_excel(path) for path in file_paths]

results = []

# 3. Calculate metrics per category
for cat in categories:
    # Construct a matrix of shape (n_annotators, n_samples) for Krippendorff's Alpha
    # Example matrix: Row 0 = Annotator 1 ratings, Row 1 = Annotator 2, etc.
    alpha_matrix = np.array([df[cat].values for df in dfs])
    
    # Krippendorff's Alpha (Ordinal accounts for distance: 0 vs 2 is worse than 0 vs 1)
    k_alpha_ordinal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='ordinal')
    k_alpha_nominal = krippendorff.alpha(reliability_data=alpha_matrix, level_of_measurement='nominal')
    
    # Construct Fleiss' Kappa table (shape: n_samples x n_categories_count)
    # Pivot so each row contains counts of [0s, 1s, 2s] assigned to that item
    sample_ratings = np.column_stack([df[cat].values for df in dfs])
    fleiss_data, _ = aggregate_raters(sample_ratings)
    f_kappa = fleiss_kappa(fleiss_data)
    
    results.append({
        "Category": cat,
        "Krippendorff_Alpha_Ordinal": round(k_alpha_ordinal, 4),
        "Krippendorff_Alpha_Nominal": round(k_alpha_nominal, 4),
        "Fleiss_Kappa": round(f_kappa, 4)
    })

# 4. Save and view summary dataframe
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Optional: Export agreement results to CSV
df_results.to_csv("inter_annotator_agreement_results.csv", index=False)

In [5]:
def build_labels_dataframe(base_dir):
    # The 14 labels formatted exactly as they appear in your raw data snippet
    LABELS = [
        "Economic",
        "Capacity_and_resources",
        "Morality",
        "Fairness_and_equality",
        "Legality_Constitutionality_and_jurisprudence",
        "Policy_prescription_and_evaluation",
        "Crime_and_punishment",
        "Security_and_defense",
        "Health_and_safety",
        "Quality_of_life",
        "Cultural_identity",
        "Public_opinion",
        "Political",
        "External_regulation_and_reputation"
    ]
    
    # Target files (add '.txt' if your files have extensions)
    target_files = ['train-labels-subtask-2.txt', 'dev-labels-subtask-2.txt']
    
    data = []
    
    # Iterate through every item in the base directory
    for lang_code in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, lang_code)
        
        # Ensure we are only looking at directories (the language folders)
        if os.path.isdir(folder_path):
            
            # Check for both train and dev files in the language folder
            for file_name in target_files:
                file_path = os.path.join(folder_path, file_name)
                
                # Check if the file exists (and also check for .txt fallback just in case)
                if not os.path.exists(file_path) and os.path.exists(file_path + ".txt"):
                    file_path += ".txt"
                    
                if os.path.exists(file_path):
                    with open(file_path, 'r', encoding='utf-8') as file:
                        for line in file:
                            line = line.strip()
                            if not line:
                                continue # Skip empty lines
                            
                            # Split by tab to separate article_id and the labels
                            parts = line.split('\t')
                            if len(parts) != 2:
                                continue # Skip malformed lines
                                
                            article_id, labels_str = parts
                            
                            # Initialize the row dictionary
                            row = {
                                'article_id': article_id,
                                'language': lang_code
                            }
                            
                            # Set default value of 0 for all 14 labels
                            for label in LABELS:
                                row[label] = 0
                                
                            # Split the comma-separated labels and set them to 1
                            current_labels = labels_str.split(',')
                            for label in current_labels:
                                label = label.strip() # Remove any accidental whitespace
                                if label in row:
                                    row[label] = 1
                                    
                            data.append(row)
                            
    # Convert the list of dictionaries into a Pandas DataFrame
    df = pd.DataFrame(data)
    
    return df

# --- Execution ---
# Replace './data_folder' with the actual path to the directory containing your language folders
base_directory = '../0_data' 

df = build_labels_dataframe(base_directory)

# Display the first few rows to verify
print(df.head())

  article_id language  Economic  Capacity_and_resources  Morality  \
0  833042063       en         0                       0         1   
1  832959523       en         1                       0         1   
2  833039623       en         0                       0         0   
3  833032367       en         0                       0         0   
4  814777937       en         1                       0         1   

   Fairness_and_equality  Legality_Constitutionality_and_jurisprudence  \
0                      1                                             1   
1                      0                                             1   
2                      0                                             1   
3                      1                                             1   
4                      1                                             0   

   Policy_prescription_and_evaluation  Crime_and_punishment  \
0                                   0                     0   
1             